# 北上广深租房市场数据分析

## 第八部分：指定预算下的整租房源选择分析

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/processed/rent_cleaned.csv")

BUDGET = 5000

df = pd.read_csv(DATA_PATH)

print("数据规模：", df.shape)
print("预算上限：", BUDGET, "元/月")

数据规模： (11978, 26)
预算上限： 5000 元/月


## 一、建立整租预算分析数据

In [2]:
# 建立整租预算分析数据
entire_df = df[
    df["type"] == "整租"
].copy()

print("清洗数据数量：", len(df))
print("整租房源数量：", len(entire_df))
print("出租类型：", entire_df["type"].unique())

清洗数据数量： 11978
整租房源数量： 11564
出租类型： <StringArray>
['整租']
Length: 1, dtype: str


## 二、检查既有异常标记

In [3]:
# 统计整租数据中的既有异常标记
anomaly_columns = [
    "is_small_entire",
    "is_large_area_conflict",
    "is_low_price_entire"
]

anomaly_counts = entire_df[
    anomaly_columns
].sum()

has_existing_anomaly = entire_df[
    anomaly_columns
].any(axis=1)

print(anomaly_counts)
print(
    "至少带有一种异常标记的整租房源：",
    has_existing_anomaly.sum()
)

is_small_entire           11
is_large_area_conflict     1
is_low_price_entire       27
dtype: int64
至少带有一种异常标记的整租房源： 38


In [4]:
# 查看预算内带有既有异常标记的整租房源
budget_anomaly_df = entire_df[
    (entire_df["rent_price_num"] <= BUDGET)
    & has_existing_anomaly
][[
    "city",
    "dist",
    "rent_area_num",
    "rent_price_num",
    "is_small_entire",
    "is_large_area_conflict",
    "is_low_price_entire"
]].copy()

budget_anomaly_df = budget_anomaly_df.sort_values(
    by=["rent_price_num", "rent_area_num"],
    ascending=[True, True]
).reset_index(drop=True)

print(
    "预算内带有既有异常标记的整租房源：",
    len(budget_anomaly_df)
)

budget_anomaly_df

预算内带有既有异常标记的整租房源： 37


,city,dist,rent_area_num,rent_price_num,is_small_entire,is_large_area_conflict,is_low_price_entire
0,广州,增城,12.0,300.0,False,False,True
1,深圳,龙岗区,26.0,320.0,False,False,True
2,深圳,坪山区,17.0,350.0,False,False,True
3,广州,天河,15.0,400.0,False,False,True
4,广州,白云,15.5,400.0,False,False,True
5,广州,荔湾,30.0,400.0,False,False,True
6,广州,增城,30.0,415.0,False,False,True
7,深圳,坪山区,16.0,430.0,False,False,True
8,深圳,龙岗区,30.0,430.0,False,False,True
9,广州,海珠,16.0,440.0,False,False,True


In [5]:
# 建立第8天专用的预算分析副本
budget_analysis_df = entire_df[
    ~has_existing_anomaly
].copy()

budget_analysis_df = budget_analysis_df.reset_index(
    drop=True
)

print("原整租房源数量：", len(entire_df))
print("预算分析房源数量：", len(budget_analysis_df))
print(
    "本次分析副本排除数量：",
    len(entire_df) - len(budget_analysis_df)
)
print(
    "分析副本中的异常标记总数：",
    budget_analysis_df[anomaly_columns].sum().sum()
)

原整租房源数量： 11564
预算分析房源数量： 11526
本次分析副本排除数量： 38
分析副本中的异常标记总数： 0


## 三、筛选预算内整租房源

In [6]:
# 筛选月租金不超过预算上限的整租房源
budget_df = budget_analysis_df[
    budget_analysis_df["rent_price_num"] <= BUDGET
].copy()

budget_df = budget_df.reset_index(
    drop=True
)

budget_share = (
    len(budget_df)
    / len(budget_analysis_df)
    * 100
)

print("预算分析房源数量：", len(budget_analysis_df))
print("预算内整租房源数量：", len(budget_df))
print("预算内房源占比：", round(budget_share, 2), "%")
print(
    "预算内最高月租金：",
    budget_df["rent_price_num"].max(),
    "元/月"
)

预算分析房源数量： 11526
预算内整租房源数量： 5895
预算内房源占比： 51.15 %
预算内最高月租金： 5000.0 元/月


## 四、比较各城市预算内可选房源

In [7]:
# 统计各城市全部及预算内整租房源数量
city_total_count = budget_analysis_df.groupby(
    "city"
).size()

city_budget_count = budget_df.groupby(
    "city"
).size()

city_budget_result = pd.DataFrame({
    "预算分析整租房源数量": city_total_count,
    "预算内整租房源数量": city_budget_count
})

city_budget_result["预算内房源占比"] = (
    city_budget_result["预算内整租房源数量"]
    / city_budget_result["预算分析整租房源数量"]
    * 100
).round(2)

city_budget_result = city_budget_result.sort_values(
    by="预算内整租房源数量",
    ascending=False
)

city_budget_result = city_budget_result.reset_index()

city_budget_result

,city,预算分析整租房源数量,预算内整租房源数量,预算内房源占比
0,广州,2874,2329,81.04
1,深圳,2670,1493,55.92
2,上海,2991,1212,40.52
3,北京,2991,861,28.79


## 五、比较各行政区预算内可选房源

In [8]:
# 统计各行政区全部及预算内整租房源数量
district_total_count = budget_analysis_df.groupby(
    ["city", "dist"]
).size()

district_budget_count = budget_df.groupby(
    ["city", "dist"]
).size()

district_budget_result = pd.DataFrame({
    "可用于分析的整租房源数量": district_total_count,
    "预算内整租房源数量": district_budget_count
})

district_budget_result = district_budget_result.fillna(
    0
)

district_budget_result["预算内整租房源数量"] = (
    district_budget_result["预算内整租房源数量"]
    .astype(int)
)

district_budget_result["预算内房源占比"] = (
    district_budget_result["预算内整租房源数量"]
    / district_budget_result["可用于分析的整租房源数量"]
    * 100
).round(2)

district_budget_result = district_budget_result.sort_values(
    by=["city", "预算内整租房源数量"],
    ascending=[True, False]
)

district_budget_result = district_budget_result.reset_index()

district_budget_result

,city,dist,可用于分析的整租房源数量,预算内整租房源数量,预算内房源占比
0,上海,浦东,791,302,38.18
1,上海,松江,261,153,58.62
2,上海,嘉定,189,134,70.90
3,上海,闵行,286,112,39.16
4,上海,宝山,167,104,62.28
5,上海,青浦,173,85,49.13
6,上海,普陀,191,73,38.22
7,上海,奉贤,57,55,96.49
8,上海,徐汇,213,51,23.94
9,上海,杨浦,108,48,44.44


In [9]:
# 查看预算内可选整租房源最多的前10个行政区
district_overall_top10 = district_budget_result.sort_values(
    by=[
        "预算内整租房源数量",
        "预算内房源占比"
    ],
    ascending=[False, False]
).head(10)

district_overall_top10 = (
    district_overall_top10
    .reset_index(drop=True)
)

district_overall_top10

,city,dist,可用于分析的整租房源数量,预算内整租房源数量,预算内房源占比
0,广州,白云,658,578,87.84
1,广州,番禺,550,479,87.09
2,深圳,宝安区,573,463,80.80
3,深圳,龙岗区,554,447,80.69
4,广州,增城,362,343,94.75
5,广州,天河,503,325,64.61
6,上海,浦东,791,302,38.18
7,广州,荔湾,215,200,93.02
8,广州,海珠,261,184,70.50
9,深圳,龙华区,316,183,57.91


In [10]:
# 分别取得每个城市预算内房源数量最多的3个行政区
district_city_sorted = district_budget_result.sort_values(
    by=[
        "city",
        "预算内整租房源数量",
        "预算内房源占比"
    ],
    ascending=[True, False, False]
)

district_city_top3 = district_city_sorted.groupby(
    "city"
).head(3)

district_city_top3 = district_city_top3.reset_index(
    drop=True
)

district_city_top3

,city,dist,可用于分析的整租房源数量,预算内整租房源数量,预算内房源占比
0,上海,浦东,791,302,38.18
1,上海,松江,261,153,58.62
2,上海,嘉定,189,134,70.90
3,北京,通州,218,140,64.22
4,北京,大兴,221,135,61.09
5,北京,丰台,319,104,32.60
6,广州,白云,658,578,87.84
7,广州,番禺,550,479,87.09
8,广州,增城,362,343,94.75
9,深圳,宝安区,573,463,80.80


## 六、统一检查分析结果

In [11]:
# 统一检查第8天的数据范围和主要结果
print("清洗数据规模：", df.shape)
print("原整租房源数量：", len(entire_df))
print("预算分析房源数量：", len(budget_analysis_df))
print("预算内整租房源数量：", len(budget_df))

print(
    "预算内出租类型：",
    budget_df["type"].unique()
)

print(
    "预算内最高月租金：",
    budget_df["rent_price_num"].max()
)

print(
    "预算分析副本异常标记总数：",
    budget_analysis_df[anomaly_columns].sum().sum()
)

print(
    "城市预算内数量合计：",
    city_budget_result["预算内整租房源数量"].sum()
)

print(
    "行政区预算内数量合计：",
    district_budget_result["预算内整租房源数量"].sum()
)

print("城市结果数量：", len(city_budget_result))
print("行政区结果数量：", len(district_budget_result))
print("总体行政区前10名数量：", len(district_overall_top10))
print("每城市行政区前3名数量：", len(district_city_top3))

清洗数据规模： (11978, 26)
原整租房源数量： 11564
预算分析房源数量： 11526
预算内整租房源数量： 5895
预算内出租类型： <StringArray>
['整租']
Length: 1, dtype: str
预算内最高月租金： 5000.0
预算分析副本异常标记总数： 0
城市预算内数量合计： 5895
行政区预算内数量合计： 5895
城市结果数量： 4
行政区结果数量： 49
总体行政区前10名数量： 10
每城市行政区前3名数量： 12


## 数据分析总结

### 数据基本情况

- 清洗数据共有11,978条住宅租房记录和26个字段。
- 其中整租房源共有11,564条。
- 检查第3天建立的三类异常标记后，仅为第8天建立11,526条记录的预算分析副本。
- 本次预算上限为5,000元/月，预算内整租房源共有5,895条，占预算分析房源的51.15%。
- 城市分析覆盖4个城市，行政区分析覆盖49个“城市＋行政区”组合。
- 原始CSV、清洗CSV和原整租数据均未修改。

### 主要分析结果

- 5,000元预算下，广州的可选整租房源最多，共2,329条，预算内房源占比为81.04%。
- 深圳有1,493条预算内整租房源，占55.92%；上海有1,212条，占40.52%；北京有861条，占28.79%。
- 城市预算内整租房源数量从高到低依次为广州、深圳、上海和北京。
- 四个城市中，广州白云的预算内整租房源最多，共578条；广州番禺为479条；深圳宝安区为463条。
- 各城市内部预算内房源数量最多的行政区分别为上海浦东、北京通州、广州白云和深圳宝安区。
- 房源数量是主要排名依据，预算内房源占比用于补充说明预算覆盖程度。
- 以上结果描述当前数据样本，不代表租房平台的实时全部市场供应。